In [1]:
!pip -q install -U transformers accelerate safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 56.7 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/My Work/Research/Chaos/Code-mixed Chaos  Multi-labeled Banglish & Bangla/Code-mixed Chaos  Multi-labeled Banglish & Bangla"

MODEL_DIR = os.path.join(BASE_PATH, "trained_bert_model")  # folder you saved earlier

LABEL_COLS = [
    "Vulgar-based",
    "Religious-Hostility",
    "Troll-based",
    "Insult-based",
    "Loathe-based",
    "Threat-based",
    "Race-based",
    "Humilaton-Based",
    "Political-Chaos",
    "Non-Toxic"
]

print("MODEL_DIR:", MODEL_DIR)
print("Exists?", os.path.exists(MODEL_DIR))
if os.path.exists(MODEL_DIR):
    print("Some files:", os.listdir(MODEL_DIR)[:10])


MODEL_DIR: /content/drive/MyDrive/My Work/Research/Chaos/Code-mixed Chaos  Multi-labeled Banglish & Bangla/Code-mixed Chaos  Multi-labeled Banglish & Bangla/trained_bert_model
Exists? True
Some files: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'special_tokens_map.json', 'vocab.txt', 'tokenizer.json', 'training_args.bin']


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Loaded on device:", device)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded on device: cuda


In [ ]:
def predict_toxicity(texts, threshold=0.5, max_length=192):
    """
    texts: str or list[str]
    returns:
      probs: np array [N, num_labels]
      preds: np array [N, num_labels]
    """
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=max_length,
        return_tensors="pt"
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.sigmoid(outputs.logits).cpu().numpy()

    preds = (probs >= threshold).astype(int)
    return probs, preds


In [ ]:
text = input("Enter text: ").strip()

probs, preds = predict_toxicity(text, threshold=0.5)

print("\n--- Results ---")
for i, label in enumerate(LABEL_COLS):
    print(f"{label:20s}  pred={preds[0][i]}  prob={probs[0][i]:.4f}")


Enter text: chudi

--- Results ---
Vulgar-based          pred=1  prob=0.7763
Religious-Hostility   pred=0  prob=0.0060
Troll-based           pred=0  prob=0.0238
Insult-based          pred=0  prob=0.1347
Loathe-based          pred=0  prob=0.0494
Threat-based          pred=0  prob=0.0236
Race-based            pred=0  prob=0.0142
Humilaton-Based       pred=1  prob=0.7090
Political-Chaos       pred=0  prob=0.0051
Non-Toxic             pred=0  prob=0.0369


In [ ]:
BASE_PATH = "/content/drive/MyDrive/My Work/Research/Chaos/Code-mixed Chaos  Multi-labeled Banglish & Bangla/Code-mixed Chaos  Multi-labeled Banglish & Bangla"

bangla_path   = f"{BASE_PATH}/Bangla.xlsx"
banglish_path = f"{BASE_PATH}/Banglish.xlsx"

In [ ]:
import pandas as pd

df_bangla   = pd.read_excel(bangla_path)
df_banglish = pd.read_excel(banglish_path)

print("Bangla shape:", df_bangla.shape)
print("Banglish shape:", df_banglish.shape)

df_bangla.head()


Bangla shape: (10234, 11)
Banglish shape: (10234, 11)


,Text,Vulgar-based,Religious-Hostility,Troll-based,Insult-based,Loathe-based,Threat-based,Race-based,Sexual-based,Political-Chaos,Non-Toxic
0,ইউসুফ সরকারকে এর চরম মূল্য দিতে হবে🤪,0,0,1,0,0,0,0,0,1,0
1,"হাসিনা ইজ লাইক আওয়ার বাঙ্গালী বয়ফ্রেইন্ড, মানু...",0,0,1,1,0,0,0,0,1,0
2,হাসিনা সরকার প্রমান করে যে মেয়েরা কখনো তাদের দ...,0,0,1,0,0,0,0,0,1,0
3,আপু তোমার কি হ্যান্ড এমব্রয়ডারি ফ্রি সিরিজ ইউট...,0,0,0,0,0,0,0,0,0,1
4,"ভাই এর লজিক টাইটানিক এর লাইফ জ্যাকেট এর মতো, দ...",0,0,1,1,0,0,0,0,0,0


In [ ]:
import pandas as pd

bangla_df = pd.read_excel(bangla_path)
banglish_df = pd.read_excel(banglish_path)

df = pd.concat([bangla_df, banglish_df], ignore_index=True)

print(df.shape)
df.head()

(20468, 11)


,Text,Vulgar-based,Religious-Hostility,Troll-based,Insult-based,Loathe-based,Threat-based,Race-based,Sexual-based,Political-Chaos,Non-Toxic
0,ইউসুফ সরকারকে এর চরম মূল্য দিতে হবে🤪,0,0,1,0,0,0,0,0,1,0
1,"হাসিনা ইজ লাইক আওয়ার বাঙ্গালী বয়ফ্রেইন্ড, মানু...",0,0,1,1,0,0,0,0,1,0
2,হাসিনা সরকার প্রমান করে যে মেয়েরা কখনো তাদের দ...,0,0,1,0,0,0,0,0,1,0
3,আপু তোমার কি হ্যান্ড এমব্রয়ডারি ফ্রি সিরিজ ইউট...,0,0,0,0,0,0,0,0,0,1
4,"ভাই এর লজিক টাইটানিক এর লাইফ জ্যাকেট এর মতো, দ...",0,0,1,1,0,0,0,0,0,0


In [ ]:
print(df.columns)

Index(['Text', 'Vulgar-based', 'Religious-Hostility', 'Troll-based',
       'Insult-based', 'Loathe-based', 'Threat-based', 'Race-based',
       'Sexual-based', 'Political-Chaos', 'Non-Toxic'],
      dtype='object')


In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    shuffle=True
)

In [ ]:
texts = test_df["Text"].tolist()

y_true = test_df[LABEL_COLS].values

KeyError: "['Humilaton-Based'] not in index"

In [ ]:
print(test_df.columns.tolist())

['Text', 'Vulgar-based', 'Religious-Hostility', 'Troll-based', 'Insult-based', 'Loathe-based', 'Threat-based', 'Race-based', 'Sexual-based', 'Political-Chaos', 'Non-Toxic']


In [ ]:
test_df = test_df.rename(columns={
    "Sexual-based": "Humilaton-Based"
})

In [ ]:
print(test_df.columns.tolist())

['Text', 'Vulgar-based', 'Religious-Hostility', 'Troll-based', 'Insult-based', 'Loathe-based', 'Threat-based', 'Race-based', 'Humilaton-Based', 'Political-Chaos', 'Non-Toxic']


In [ ]:
texts = test_df["Text"].tolist()

y_true = test_df[LABEL_COLS].values

In [ ]:
print(texts[:3])
print(y_true.shape)

['Shala dhorbo tore', 'কঠিন এই শোক মেনে নেওয়া', 'সরকারের চেয়ে হাজার গুন্ জানোয়ার হলো এই নাম ধরি আলেম সমাজ যারা ইসলাম কে ধ্বংস করছে.']
(3071, 10)


In [ ]:
import numpy as np
import torch
from tqdm import tqdm

def predict_toxicity_batch(
    texts,
    batch_size=32,
    threshold=0.5,
    max_length=192
):
    all_probs = []

    model.eval()

    with torch.no_grad():

        for i in tqdm(range(0, len(texts), batch_size)):

            batch_texts = texts[i:i+batch_size]

            enc = tokenizer(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt"
            )

            enc = {k: v.to(device) for k, v in enc.items()}

            outputs = model(**enc)

            probs = torch.sigmoid(outputs.logits)

            all_probs.append(probs.cpu())

            # Free GPU memory
            del outputs, probs, enc
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    all_probs = torch.cat(all_probs).numpy()

    preds = (all_probs >= threshold).astype(int)

    return all_probs, preds

In [ ]:
probs, preds = predict_toxicity_batch(
    texts,
    batch_size=8
)

100%|██████████| 384/384 [00:13<00:00, 28.85it/s]


In [ ]:
np.save("mbert_preds.npy", preds)
np.save("ground_truth.npy", y_true)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
import torch

MODEL_DIR = os.path.join(BASE_PATH, "trained_xlmr_model")   # <-- Change if needed

print("Loading:", MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

print("Loaded successfully on", device)

Loading: /content/drive/MyDrive/My Work/Research/Chaos/Code-mixed Chaos  Multi-labeled Banglish & Bangla/Code-mixed Chaos  Multi-labeled Banglish & Bangla/trained_xlmr_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded successfully on cuda


In [ ]:
probs_xlmr, preds_xlmr = predict_toxicity_batch(
    texts,
    batch_size=8,
    threshold=0.5
)

print(preds_xlmr.shape)

100%|██████████| 384/384 [00:12<00:00, 31.14it/s]

(3071, 10)


In [ ]:
import numpy as np

np.save("xlmr_preds.npy", preds_xlmr)

print("Saved XLM-R predictions.")

Saved XLM-R predictions.


In [ ]:
mbert_preds = np.load("mbert_preds.npy")
xlmr_preds = np.load("xlmr_preds.npy")
ground_truth = np.load("ground_truth.npy")

print("Ground Truth :", ground_truth.shape)
print("mBERT        :", mbert_preds.shape)
print("XLM-R        :", xlmr_preds.shape)

Ground Truth : (3071, 10)
mBERT        : (3071, 10)
XLM-R        : (3071, 10)


McNeymar Test

In [ ]:
!pip -q install statsmodels tabulate

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

In [ ]:
ground_truth = np.load("ground_truth.npy")
mbert_preds = np.load("mbert_preds.npy")
xlmr_preds = np.load("xlmr_preds.npy")

print("Ground Truth :", ground_truth.shape)
print("mBERT        :", mbert_preds.shape)
print("XLM-R        :", xlmr_preds.shape)

Ground Truth : (3071, 10)
mBERT        : (3071, 10)
XLM-R        : (3071, 10)


In [ ]:
LABEL_COLS = [
    "Vulgar-based",
    "Religious-Hostility",
    "Troll-based",
    "Insult-based",
    "Loathe-based",
    "Threat-based",
    "Race-based",
    "Humilation-Based",
    "Political-Chaos",
    "Non-Toxic"
]

In [ ]:
results = []

for i, label in enumerate(LABEL_COLS):

    gt = ground_truth[:, i]

    mbert = mbert_preds[:, i]
    xlmr = xlmr_preds[:, i]

    mbert_correct = (mbert == gt)
    xlmr_correct = (xlmr == gt)

    b = np.sum(mbert_correct & (~xlmr_correct))
    c = np.sum((~mbert_correct) & xlmr_correct)

    table = [[0, b],
             [c, 0]]

    result = mcnemar(table, exact=True)

    results.append({
        "Label": label,
        "b": b,
        "c": c,
        "Statistic": result.statistic,
        "p-value": result.pvalue,
        "Significant": "Yes" if result.pvalue < 0.05 else "No"
    })

df = pd.DataFrame(results)

df

,Label,b,c,Statistic,p-value,Significant
0,Vulgar-based,88,66,66.0,0.090277,No
1,Religious-Hostility,37,40,37.0,0.819893,No
2,Troll-based,62,104,62.0,0.001388,Yes
3,Insult-based,86,116,86.0,0.041041,Yes
4,Loathe-based,69,80,69.0,0.412743,No
5,Threat-based,76,93,76.0,0.218294,No
6,Race-based,44,31,31.0,0.165428,No
7,Humilation-Based,107,85,85.0,0.129419,No
8,Political-Chaos,69,47,47.0,0.050730,No
9,Non-Toxic,80,122,80.0,0.003809,Yes


In [ ]:
df.to_csv("McNemar_results.csv", index=False)

print(df)

                 Label    b    c  Statistic   p-value Significant
0         Vulgar-based   88   66       66.0  0.090277          No
1  Religious-Hostility   37   40       37.0  0.819893          No
2          Troll-based   62  104       62.0  0.001388         Yes
3         Insult-based   86  116       86.0  0.041041         Yes
4         Loathe-based   69   80       69.0  0.412743          No
5         Threat-based   76   93       76.0  0.218294          No
6           Race-based   44   31       31.0  0.165428          No
7     Humilation-Based  107   85       85.0  0.129419          No
8      Political-Chaos   69   47       47.0  0.050730          No
9            Non-Toxic   80  122       80.0  0.003809         Yes


In [ ]:
mbert_correct = (mbert_preds == ground_truth)
xlmr_correct = (xlmr_preds == ground_truth)

b = np.sum(mbert_correct & (~xlmr_correct))
c = np.sum((~mbert_correct) & xlmr_correct)

table = [[0, b],
         [c, 0]]

result = mcnemar(table, exact=True)

print("="*60)
print("Overall McNemar Test")
print("="*60)
print("b =", b)
print("c =", c)
print("Statistic =", result.statistic)
print("p-value =", result.pvalue)

if result.pvalue < 0.05:
    print("\n✓ Significant Difference (p < 0.05)")
else:
    print("\n✗ No Significant Difference")

Overall McNemar Test
b = 718
c = 784
Statistic = 718.0
p-value = 0.09347527580138437

✗ No Significant Difference
